# Synthetic Tabular Data Generation - Staged Optimization Driver

**Enhanced notebook for synthetic tabular data generation with staged hyperparameter optimization.**

This notebook provides a staged optimization workflow with:
- Section 1: Setup and imports
- Section 2: Data loading, preprocessing, and EDA
- Section 3: Demo models with default parameters
- Section 4: **Staged Hyperparameter Optimization** (NEW)
  - 4.1: Configuration
  - 4.2: Pilot phase (15 trials) with time estimates
  - 4.3: Continuation options
  - 4.4: Diminishing returns analysis
  - 4.5: Additional batches (optional)
- Section 5: Final models with best parameters

Based on STG-Driver-breast-cancer.ipynb with staged optimization enhancements.

## 1 Setup and Configuration

In [1]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
import sys
sys.path.insert(0, "/home/ec2-user/SageMaker/tableGenCompare")

from setup import *

# Refresh session timestamp to current date
refresh_session_timestamp()

# ============================================================================
# FRESH START CONTROL - Set to True to wipe all checkpoints and start over
# ============================================================================
FRESH_START = True   # <-- Change to True to clear ALL saved progress
# ============================================================================

print("SETUP MODULE IMPORTED SUCCESSFULLY!")
print(f"FRESH_START = {FRESH_START}")
print("=" * 60)

[OK] Essential data science libraries imported successfully!
[OK] Model registry loaded from src/models/registry.py
[OK] Batch training module loaded from src/models/batch_training.py
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py
[OK] Staged optimization module loaded from src/models/staged_optimization.py
[CONFIG] Session timestamp: 2026-03-11
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
[OK] Optuna objective functions for PATE-GAN and MEDGAN loaded (Phase 5)
Detected sklearn 1.7.2 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.7.2
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementa

## 2 Data Loading and Pre-processing

### 2.1 Configuration

**USER ATTENTION NEEDED**: Edit the `NOTEBOOK_CONFIG` dictionary below to match your dataset.

In [2]:
# Code Chunk ID: CHUNK_2_1_1_A - NOTEBOOK CONFIGURATION
# ============================================================================
# USER CONFIGURATION - Edit ONLY this block for your dataset
# ============================================================================

NOTEBOOK_CONFIG = {
    # ========== REQUIRED: Dataset Settings ==========
    "data_file": "data/Pakistani_Diabetes_Dataset.csv",  # Path to your CSV file
    "target_column": "Outcome",                 # Target/outcome column name

    # ========== OPTIONAL: Dataset Metadata ==========
    "dataset_name": "Pakistani Diabetes Dataset",      # Display name
    "dataset_identifier_override": None,          # Override auto-detected ID (or None)

    # ========== OPTIONAL: Column Configuration ==========
    # Binary 0/1 flags in this dataset (7 features):
    "categorical_columns": ["Gender", "Rgn ", "his", "vision", "dipsia", "uria", "neph"],
    "task_type": "classification",

    # ========== OPTIONAL: Fairness Evaluation ==========
    # Protected attribute for fairness metrics (use cleaned column name after preprocessing).
    # Gender is binary (0/1) and survives preprocessing as-is.
    "protected_col": "gender",

    # ========== OPTIONAL: Data Subsetting ==========
    "use_row_subset": False,                      # 912 rows - small enough to use all
    "sample_n": 500,                              # Number of rows to sample
    "sample_random_state": 42,                    # Random seed for reproducibility

    # ========== OPTIONAL: Missing Data Handling ==========
    "missing_strategy": "none",                   # No missing values in this dataset
    "mice_max_iter": 10,                          # Max iterations for MICE imputation

    # ========== OPTIONAL: Model Selection ==========
    "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "pategan", "medgan"],     # "all" or list like ["ctgan", "tvae"]
    # "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "ganeraid", "pategan", "medgan"],  

    # ========== OPTIONAL: Tuning Configuration ==========
    "tuning_mode": "full",                        # "smoke" (quick) | "full" (thorough)
    "n_trials_smoke": 15,                          # Trials for smoke testing / pilot phase
}

print("NOTEBOOK_CONFIG set successfully!")
print(f"  Data file: {NOTEBOOK_CONFIG['data_file']}")
print(f"  Target column: {NOTEBOOK_CONFIG['target_column']}")
print(f"  Protected column: {NOTEBOOK_CONFIG.get('protected_col', None)}")
print(f"  Models to run: {NOTEBOOK_CONFIG['models_to_run']}")
print(f"  Tuning mode: {NOTEBOOK_CONFIG['tuning_mode']}")

NOTEBOOK_CONFIG set successfully!
  Data file: data/Pakistani_Diabetes_Dataset.csv
  Target column: Outcome
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  Tuning mode: full


### 2.2 Load and Preprocess Data

In [3]:
# Code Chunk ID: CHUNK_2_1_2_A - LOAD AND PREPROCESS DATA
# ============================================================================
# This uses the config-driven preprocessing pipeline
# ============================================================================

if not FRESH_START and 'checkpoint' in dir() and hasattr(checkpoint, 'exists') and checkpoint.exists('section_2'):
    saved = checkpoint.load('section_2')
    data = saved['data']
    original_data = saved['original_data']
    target_column = saved['target_column']
    DATASET_IDENTIFIER = saved['DATASET_IDENTIFIER']
    categorical_columns = saved['categorical_columns']
    metadata = saved['metadata']
    models_to_run = saved['models_to_run']
    RUN_MODE = saved['RUN_MODE']
    TARGET_COLUMN = target_column
    print("[RESUME] Loaded Section 2 from checkpoint")
else:
    # Load and preprocess data using the config
    (
        data,                  # Processed DataFrame
        original_data,         # Copy for reference
        target_column,         # Target column name (cleaned)
        DATASET_IDENTIFIER,    # Dataset identifier for results paths
        categorical_columns,   # List of categorical columns
        metadata               # Full preprocessing metadata
    ) = load_and_preprocess_from_config(NOTEBOOK_CONFIG)

    # Set aliases for backward compatibility with existing notebook cells
    TARGET_COLUMN = target_column

    # Get models to run based on config
    models_to_run = get_models_to_run(NOTEBOOK_CONFIG)

    # Set RUN_MODE for backward compatibility with Section 4 tuning cells
    RUN_MODE = "debug" if NOTEBOOK_CONFIG['tuning_mode'] == "smoke" else "full"

# Initialize checkpoint system (now that DATASET_IDENTIFIER is known)
checkpoint = SectionCheckpoint(DATASET_IDENTIFIER)

# If FRESH_START, wipe all checkpoints and optimization studies
if FRESH_START:
    n_removed = checkpoint.clear_all()
    print(f"[FRESH START] Cleared {n_removed} checkpoint(s) and optimization studies")

existing = checkpoint.list_checkpoints()
if existing:
    print(f"[CHECKPOINT] Found {len(existing)} existing checkpoint(s): {existing}")

# Save Section 2 checkpoint (idempotent - overwrites if exists)
if not checkpoint.exists('section_2'):
    checkpoint.save('section_2', {
        'data': data, 'original_data': original_data,
        'target_column': target_column, 'DATASET_IDENTIFIER': DATASET_IDENTIFIER,
        'categorical_columns': categorical_columns, 'metadata': metadata,
        'models_to_run': models_to_run, 'RUN_MODE': RUN_MODE,
    })

print()
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"  Dataset identifier: {DATASET_IDENTIFIER}")
print(f"  Processed shape: {data.shape}")
print(f"  Target column: {target_column}")
print(f"  Task type: {metadata['task_type']}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Models to run: {models_to_run}")
print(f"  RUN_MODE: {RUN_MODE}")
print(f"  Session: {SESSION_TIMESTAMP}")
print(f"  Results path: {get_results_path(DATASET_IDENTIFIER, 2)}")
print("=" * 60)

[LOAD] Loading data from: data/Pakistani_Diabetes_Dataset.csv
[LOAD] Loaded 912 rows, 19 columns
[PREPROCESS] Starting preprocessing pipeline
[PREPROCESS] Input shape: (912, 19)
[PREPROCESS] Categorical columns: ['gender', 'rgn', 'his', 'vision', 'dipsia', 'uria', 'neph']
[PREPROCESS] Task type: classification
[PREPROCESS] Final shape: (912, 19)
[PREPROCESS] Dataset identifier: pakistani-diabetes-dataset
[FLUSH] Removed 14 pickle file(s) from results/pakistani-diabetes-dataset/optimization_studies
[FRESH START] Cleared 17 checkpoint(s) and optimization studies

PREPROCESSING COMPLETE
  Dataset identifier: pakistani-diabetes-dataset
  Processed shape: (912, 19)
  Target column: outcome
  Task type: classification
  Categorical columns: ['gender', 'rgn', 'his', 'vision', 'dipsia', 'uria', 'neph']
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  RUN_MODE: full
  Session: 2026-03-11
  Results path: results/pakistani-diabetes-dataset/2026-03-1

### 2.3 Exploratory Data Analysis (EDA)

Comprehensive EDA with automated file export to results directory.

In [4]:
# Code Chunk ID: CHUNK_2_1_EDA - COMPREHENSIVE EDA
# ============================================================================
# Run comprehensive EDA with single function call
# ============================================================================

from src.data.eda import run_comprehensive_eda

eda_results = run_comprehensive_eda(
    data=data,
    target_column=target_column,
    dataset_identifier=DATASET_IDENTIFIER,
    dataset_name=NOTEBOOK_CONFIG.get('dataset_name'),
    categorical_columns=categorical_columns,
    verbose=True
)

# Update categorical_columns from EDA results (in case auto-detected)
categorical_columns = eda_results['categorical_columns']

print(f"\nEDA Results Summary:")
print(f"  Files generated: {len(eda_results['files_generated'])}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Balance ratio: {eda_results.get('balance_ratio', 'N/A')}")

COMPREHENSIVE EXPLORATORY DATA ANALYSIS
Dataset: Pakistani Diabetes Dataset
Target column: outcome
Results path: results/pakistani-diabetes-dataset/2026-03-11/Section-2

[1/6] Dataset Overview...
   Dataset Name.................. Pakistani Diabetes Dataset
   Shape......................... 912 rows x 19 columns
   Memory Usage.................. 0.13 MB
   Total Missing Values.......... 0
   Missing Percentage............ 0.00%
   Duplicate Rows................ 2
   Numeric Columns............... 19
   Categorical Columns........... 0

[2/6] Column Analysis...
   Saved: column_analysis.csv (19 columns)

[3/6] Target Variable Analysis...
   Saved: target_analysis.csv
   Saved: target_balance_metrics.csv
   Balance Ratio: 0.877 (Balanced)

[4/6] Feature Distributions...
   Saved: 3 distribution file(s) (18 features)

[5/6] Correlation Analysis...
   Saved: correlation_heatmap.png
   Saved: correlation_matrix.csv
   Saved: target_correlations.csv (18 features)

[6/6] Configuration Validati

## 3 Demo All Models with Default Parameters

### 3.1 Batch Model Training

In [5]:
# Code Chunk ID: CHUNK_3_1_BATCH
# ============================================================================
# SECTION 3.1 - BATCH MODEL TRAINING
# Train all models configured in NOTEBOOK_CONFIG['models_to_run']
# ============================================================================

from src.models.batch_training import train_models_batch, extract_synthetic_data_to_globals

# Run batch training for all configured models (checkpoint resumes completed models)
training_results = train_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    verbose=True,
    checkpoint=checkpoint
)

# Extract synthetic_data_* variables for Section 3.2 compatibility
created_vars = extract_synthetic_data_to_globals(training_results, globals())
print(f"\nCreated variables: {created_vars}")

# Also create model_* variables for backward compatibility
for model_name, result in training_results.items():
    if result['status'] == 'success' and result.get('model') is not None:
        globals()[f'{model_name}_model'] = result['model']


BATCH MODEL TRAINING
Models to train: 7
Dataset shape: (912, 19)
Target column: outcome
Samples to generate: 912
GPU available: NVIDIA A10G (22.1 GB)
Device assignments:
  - CTGAN: cuda
  - TVAE: cuda
  - CopulaGAN: cuda
  - CTABGAN: cuda
  - CTABGAN+: cuda
  - PATE-GAN: cuda
  - MEDGAN: cuda


[1/7] Training CTGAN...
--------------------------------------------------
  Device: cuda
  Training CTGAN...


Gen. (-1.47) | Discrim. (-0.01): 100%|██████████| 300/300 [00:12<00:00, 24.69it/s]


  Generating 912 synthetic samples...
  [OK] CTGAN completed in 20.08s
  Synthetic data shape: (912, 19)

[2/7] Training TVAE...
--------------------------------------------------
  Device: cuda
  Training TVAE...
  Generating 912 synthetic samples...
[OK] Target integrity functions loaded from src/data/target_integrity.py
  [OK] TVAE completed in 12.25s
  Synthetic data shape: (912, 19)

[3/7] Training CopulaGAN...
--------------------------------------------------
  Device: cuda
  Training CopulaGAN...
  Generating 912 synthetic samples...
  [OK] CopulaGAN completed in 14.73s
  Synthetic data shape: (912, 19)

[4/7] Training CTABGAN...
--------------------------------------------------
  Device: cuda
  Training CTABGAN...


100%|██████████| 300/300 [00:24<00:00, 12.20it/s]


Finished training in 26.44079852104187  seconds.
  Generating 912 synthetic samples...
  [OK] CTABGAN completed in 26.51s
  Synthetic data shape: (912, 19)

[5/7] Training CTABGAN+...
--------------------------------------------------
  Device: cuda
  Training CTABGAN+...


100%|██████████| 400/400 [00:49<00:00,  8.08it/s]


Finished training in 51.357704401016235  seconds.
  Generating 912 synthetic samples...
  [OK] CTABGAN+ completed in 51.45s
  Synthetic data shape: (912, 19)

[6/7] Training PATE-GAN...
--------------------------------------------------
  Device: cuda
  Training PATE-GAN...
  Generating 912 synthetic samples...
  [OK] PATE-GAN completed in 12.13s
  Synthetic data shape: (912, 19)

[7/7] Training MEDGAN...
--------------------------------------------------
  Device: cuda
  Training MEDGAN...
  Generating 912 synthetic samples...
  [OK] MEDGAN completed in 12.95s
  Synthetic data shape: (912, 19)

BATCH TRAINING SUMMARY
Total models: 7
Successful: 7
Failed: 0

Successful models:
  - CTGAN: 20.08s
  - TVAE: 12.25s
  - CopulaGAN: 14.73s
  - CTABGAN: 26.51s
  - CTABGAN+: 51.45s
  - PATE-GAN: 12.13s
  - MEDGAN: 12.95s

Created variables: ['synthetic_data_ctgan', 'synthetic_data_tvae', 'synthetic_data_copulagan', 'synthetic_data_ctabgan', 'synthetic_data_ctabganplus', 'synthetic_data_pategan'

### 3.2 Batch Evaluation

In [6]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3.2 - BATCH EVALUATION FOR ALL TRAINED MODELS
# ============================================================================

print("SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

if checkpoint.exists('section_3.2'):
    section3_results = checkpoint.load('section_3.2')['results']
    print("[RESUME] Loaded Section 3.2 evaluation from checkpoint")
else:
    section3_results = evaluate_all_available_models(
        section_number=3,
        scope=globals(),
        models_to_evaluate=None,
        real_data=None,
        target_col=None,
        protected_col=NOTEBOOK_CONFIG.get("protected_col")
    )
    checkpoint.save('section_3.2', {'results': section3_results})

if section3_results:
    print(f"\nSECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"Evaluated {len(section3_results)} models successfully")
    
    # Show ranking by quality score
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))
    
    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\nRANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\nNo models available for evaluation")

SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: pakistani-diabetes-dataset
[INFO] Target column: outcome
[INFO] Protected column: None (fairness metrics skipped)
[INFO] MIA evaluation: OFF
[INFO] Variable pattern: standard
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/pakistani-diabetes-dataset/2026-03-11/Section-3/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.793

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Over

## 4 Staged Hyperparameter Optimization

This section uses a staged approach to hyperparameter optimization:

- **Smoke mode** (`tuning_mode="smoke"`): Runs 10 pilot trials per model, then displays
  time-budget recommendations (how many trials fit in 1 hour, how long 20 trials would take).
  Section 5 uses the pilot results directly.
- **Full mode** (`tuning_mode="full"`): Runs a pilot phase, displays time estimates, then
  presents three continuation strategies in cell 4.3.  The user reviews the estimates and
  **uncomments one option** before running the cell.

### 4.1 Configuration

Create the `StagedOptimizationManager`.  `TUNING_MODE` and `PILOT_TRIALS` are derived
from `NOTEBOOK_CONFIG` so the same code works for both smoke and full runs.

In [7]:
# Code Chunk ID: CHUNK_4_1_CONFIG
# ============================================================================
# SECTION 4.1 - STAGED OPTIMIZATION CONFIGURATION
# ============================================================================

from src.models.staged_optimization import (
    StagedOptimizationManager,
    StagedOptimizationConfig,
    flush_previous_runs
)

# Flush optimization studies if FRESH_START is set
if FRESH_START:
    flush_previous_runs(DATASET_IDENTIFIER)

# Derive tuning mode and pilot size from NOTEBOOK_CONFIG
TUNING_MODE = NOTEBOOK_CONFIG.get("tuning_mode", "smoke")
PILOT_TRIALS = 10 if TUNING_MODE == "smoke" else NOTEBOOK_CONFIG.get("n_trials_smoke", 10)

# Configure staged optimization
staged_config = StagedOptimizationConfig(
    pilot_trials=PILOT_TRIALS,
    verbose_level=1,           # 0=silent, 1=concise, 2=detailed
    persistence_dir=f"results/{DATASET_IDENTIFIER}/optimization_studies",
    run_mode=RUN_MODE,         # "debug" or "full"
    random_state=42,
    continue_on_error=True
)

# Create the manager
manager = StagedOptimizationManager(
    data=data,
    target_column=target_column,
    categorical_columns=categorical_columns,
    dataset_identifier=DATASET_IDENTIFIER,
    config=staged_config
)

print("Staged Optimization Manager created!")
print(f"  Tuning mode: {TUNING_MODE}")
print(f"  Pilot trials: {staged_config.pilot_trials}")
print(f"  Run mode: {staged_config.run_mode}")
print(f"  Persistence dir: {staged_config.persistence_dir}")
print(f"  FRESH_START: {FRESH_START}")

[FLUSH] No previous studies found in results/pakistani-diabetes-dataset/optimization_studies — starting clean
Staged Optimization Manager created!
  Tuning mode: full
  Pilot trials: 15
  Run mode: full
  Persistence dir: results/pakistani-diabetes-dataset/optimization_studies
  FRESH_START: True


### 4.2 Run Pilot Phase

Run a pilot phase to establish time estimates.

- **Smoke mode**: 10 trials + smoke recommendations table (trials in 1 hr, time for 20 trials).
- **Full mode**: Pilot trials + time estimates for planning continuation.

In [8]:
# Code Chunk ID: CHUNK_4_2_PILOT
# ============================================================================
# SECTION 4.2 - RUN PILOT PHASE
# ============================================================================

# Run pilot phase (uses PILOT_TRIALS from cell 4.1)
pilot_states = manager.run_pilot(
    models_to_run=models_to_run
)

# Display time estimates
print("\n" + "="*60)
print("PILOT PHASE COMPLETE - TIME ESTIMATES")
print("="*60)

time_estimates = manager.get_time_estimates()
if len(time_estimates) > 0:
    print(time_estimates.to_string(index=False))
else:
    print("No time estimates available")

# Show best scores so far
print("\nBest scores after pilot:")
for model_name, state in pilot_states.items():
    print(f"  {model_name}: {state.best_score:.4f} ({state.total_trials} trials)")

# Smoke mode: show budget recommendations
if TUNING_MODE == "smoke":
    print("\n" + "="*60)
    print("SMOKE MODE - TIME BUDGET RECOMMENDATIONS")
    print("="*60)
    smoke_recs = manager.get_smoke_recommendations()
    print(smoke_recs.to_string(index=False))
    print("\nTo run a full optimization, set tuning_mode='full' in NOTEBOOK_CONFIG")
    print("and use the recommended_pilot column to guide n_trials_smoke.")

[I 2026-03-11 18:04:37,329] A new study created in memory with name: ctgan_hpo_pakistani-diabetes-dataset



STAGED OPTIMIZATION - PILOT PHASE
Models to optimize: 7
Pilot trials per model: 15
Dataset identifier: pakistani-diabetes-dataset


[PILOT] Optimizing CTGAN...
--------------------------------------------------


Gen. (-0.17) | Discrim. (-0.08): 100%|██████████| 700/700 [00:42<00:00, 16.62it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5298
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7752
[CHART] Combined Score: 0.6280 (Similarity: 0.5298, Accuracy: 0.7752)
[ctgan] Trial 1: Combined Score: 0.6280 (Similarity: 0.5298, Accuracy: 0.7752) Best Score so far: 0.6280


Gen. (-0.73) | Discrim. (-0.17): 100%|██████████| 650/650 [00:38<00:00, 16.69it/s]


...

Finished training in 95.08019185066223  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5491
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9704
[CHART] Combined Score: 0.7176 (Similarity: 0.5491, Accuracy: 0.9704)
[ctabganplus] Trial 13: Combined Score: 0.7176 (Similarity: 0.5491, Accuracy: 0.9704) Best Score so far: 0.7522


100%|██████████| 150/150 [01:04<00:00,  2.32it/s]


Finished training in 66.1243245601654  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5438
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9644
[CHART] Combined Score: 0.7120 (Similarity: 0.5438, Accuracy: 0.9644)
[ctabganplus] Trial 14: Combined Score: 0.7120 (Similarity: 0.5438, Accuracy: 0.9644) Best Score so far: 0.7522


100%|██████████| 500/500 [03:34<00:00,  2.33it/s]


Finished training in 216.46571516990662  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5765
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9786
[CHART] Combined Score: 0.7373 (Similarity: 0.5765, Accuracy: 0.9786)
[ctabganplus] Trial 15: Combined Score: 0.7373 (Similarity: 0.5765, Accuracy: 0.9786) Best Score so far: 0.7522
  [OK] CTABGAN+: 15 trials in 1801.1s
  Best score: 0.7522

[PILOT] Optimizing PATE-GAN...
--------------------------------------------------
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.3251
[OK] TRTS Evaluation: 2 scenarios, Average: 0.2336
[CHART] Combined Score: 0.2885 (Similarity: 0.3251, Accuracy: 0.2336)
[pategan] Trial 1: Combined Score: 0.2885 (Similarity: 0.3251, Accuracy: 0.2336) Best Score so far: 0.2885
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity A

In [9]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS
# ============================================================================
print("DIMINISHING RETURNS ANALYSIS")
print("="*60)

convergence_report = manager.get_diminishing_returns_report()

if len(convergence_report) > 0:
    print(convergence_report.to_string(index=False))
    
    print("\nInterpretation:")
    print("-" * 40)
    for _, row in convergence_report.iterrows():
        print(f"  {row['model']}: {row['recommendation']}")
        if row['has_plateaued']:
            print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
        else:
            print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
else:
    print("No convergence data available")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued         recommendation
      ctgan      15    0.690054          0.000589           0.062095           True Stop - plateau reached
       tvae      15    0.792095          0.004373           0.102928           True Stop - plateau reached
  copulagan      15    0.765058          0.000000           0.292355           True Stop - plateau reached
    ctabgan      15    0.746152          0.000000           0.025489           True Stop - plateau reached
ctabganplus      15    0.752234          0.000000           0.012183           True Stop - plateau reached
    pategan      15    0.415606          0.000000           0.127130           True Stop - plateau reached
     medgan      15    0.357358          0.000000           0.042690           True Stop - plateau reached

Interpretation:
----------------------------------------
  ctgan: Stop - plateau reached
    -> Model has plateaue

### 4.3 Continuation (Full Mode Only)

**Full mode only** - Review the pilot results and time estimates above, then
uncomment **one** of the three options below and adjust the values before running.

- **Option (i)**: Common trial count for all models
- **Option (ii)**: Per-model trial counts
- **Option (iii)**: Time-based budget (minutes per model)

Models not in `models_to_run` are silently ignored, so listing all 8 is safe.

In [10]:
# Code Chunk ID: CHUNK_4_3_CONTINUE
# ============================================================================
# SECTION 4.3 - CONTINUATION (Full mode only - choose ONE option)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping continuation - using pilot results for Section 5.")

else:
    # Choose ONE option below, then run this cell.

    # OPTION (i): Common trials for all models
    # continuation_states = manager.continue_optimization(additional_trials=30)

    # OPTION (ii): Per-model trials - adjust counts per model
    continuation_states = manager.continue_optimization(
        trials_per_model={
            "ctgan": 30,
            # "tvae": 30,
            # "copulagan": 30,
            # "ctabgan": 30,
            "ctabganplus": 15,
            # "ganeraid": 30,
            # "pategan": 30,
            "medgan": 50,
        }
    )

    # OPTION (iii): Time-based budget - minutes per model
    # continuation_states = manager.continue_optimization(
    #     time_budget_minutes={
    #         "ctgan": 60,
    #         "tvae": 60,
    #         "copulagan": 60,
    #         "ctabgan": 60,
    #         "ctabganplus": 60,
    #         "ganeraid": 60,
    #         "pategan": 60,
    #         "medgan": 60,
    #     }
    # )

    print("[FULL MODE] Continuation completed.")


STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 30 additional trials
  ctabganplus: 15 additional trials
  medgan: 50 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 15 existing trials


Gen. (-0.04) | Discrim. (-0.08): 100%|██████████| 750/750 [00:37<00:00, 20.09it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5449
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8827
[CHART] Combined Score: 0.6800 (Similarity: 0.5449, Accuracy: 0.8827)
[ctgan] Trial 16: Combined Score: 0.6800 (Similarity: 0.5449, Accuracy: 0.8827) Best Score so far: 0.6901


Gen. (-1.16) | Discrim. (0.06): 100%|██████████| 900/900 [00:45<00:00, 19.86it/s] 


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5567
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8794
[CHART] Combined Score: 0.6857 (Similarity: 0.5567, Accuracy: 0.8794)
[ctgan] Trial 17: Combined Score: 0.6857 (Similarity: 0.5567, Accuracy: 0.8794) Best Score so far: 0.6901


...

Finished training in 301.7869381904602  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5865
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9786
[CHART] Combined Score: 0.7433 (Similarity: 0.5865, Accuracy: 0.9786)
[ctabganplus] Trial 28: Combined Score: 0.7433 (Similarity: 0.5865, Accuracy: 0.9786) Best Score so far: 0.7545


100%|██████████| 800/800 [05:43<00:00,  2.33it/s]


Finished training in 344.92515754699707  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6203
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9814
[CHART] Combined Score: 0.7647 (Similarity: 0.6203, Accuracy: 0.9814)
[ctabganplus] Trial 29: Combined Score: 0.7647 (Similarity: 0.6203, Accuracy: 0.9814) Best Score so far: 0.7647


100%|██████████| 900/900 [06:26<00:00,  2.33it/s]


Finished training in 387.8487012386322  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6076
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9814
[CHART] Combined Score: 0.7571 (Similarity: 0.6076, Accuracy: 0.9814)
[ctabganplus] Trial 30: Combined Score: 0.7571 (Similarity: 0.6076, Accuracy: 0.9814) Best Score so far: 0.7647
  [OK] CTABGAN+: 15 trials in 2749.9s
  Best score: 0.7647

[CONTINUE] Optimizing MEDGAN...
--------------------------------------------------
  Resuming from 15 existing trials
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.3961
[OK] TRTS Evaluation: 2 scenarios, Average: 0.2363
[CHART] Combined Score: 0.3322 (Similarity: 0.3961, Accuracy: 0.2363)
[medgan] Trial 16: Combined Score: 0.3322 (Similarity: 0.3961, Accuracy: 0.2363) Best Score so far: 0.3574
[TARGET] Enhanced objective function using target 

In [11]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS (post-continuation)
# ============================================================================

if TUNING_MODE == "full":
    print("DIMINISHING RETURNS ANALYSIS")
    print("="*60)

    convergence_report = manager.get_diminishing_returns_report()

    if len(convergence_report) > 0:
        print(convergence_report.to_string(index=False))

        print("\nInterpretation:")
        print("-" * 40)
        for _, row in convergence_report.iterrows():
            print(f"  {row['model']}: {row['recommendation']}")
            if row['has_plateaued']:
                print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
            else:
                print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
    else:
        print("No convergence data available")
else:
    print("[SMOKE MODE] Skipping post-continuation analysis.")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued                         recommendation
      ctgan      45    0.698564          0.005454           0.070605           True                 Stop - plateau reached
       tvae      15    0.792095          0.004373           0.102928           True                 Stop - plateau reached
  copulagan      15    0.765058          0.000000           0.292355           True                 Stop - plateau reached
    ctabgan      15    0.746152          0.000000           0.025489           True                 Stop - plateau reached
ctabganplus      30    0.764708          0.013306           0.024658          False Consider stopping - minor improvements
    pategan      15    0.415606          0.000000           0.127130           True                 Stop - plateau reached
     medgan      65    0.526862          0.000000           0.212194           True                 Stop - pla

### 4.5 Additional Batches (Full Mode, Optional)

If the diminishing returns analysis suggests continuing, uncomment one option below.
You can re-run this cell multiple times.

In [13]:
# Code Chunk ID: CHUNK_4_5_ADDITIONAL

# ============================================================================
# SECTION 4.5 - ADDITIONAL BATCHES (Full mode only - optional, can repeat)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping additional batches.")
else:
    # ---- Uncomment ONE option below, adjust values, then run this cell ----

    # OPTION (i): Common trials for all models
    # additional_states = manager.continue_optimization(additional_trials=20)

    # OPTION (ii): Per-model trials - adjust counts per model
    # additional_states = manager.continue_optimization(
    #     trials_per_model={
    #         'ctgan': 20,
    #         'tvae': 20,
    #         'copulagan': 20,
    #         'ctabgan': 20,
    #         'ctabganplus': 20,
    #         'ganeraid': 20,
    #         'pategan': 20,
    #         'medgan': 20,
    #     }
    # )

    # OPTION (iii): Time-based budget - minutes per model
    additional_states = manager.continue_optimization(
        time_budget_minutes={
            'ctgan': 30,
            'tvae': 30,
            # 'copulagan': 30,
            # 'ctabgan': 30,
            'ctabganplus': 30,
            # 'ganeraid': 30,
            # 'pategan': 30,
            # 'medgan': 30,
        }
    )

    # print("\nUpdated convergence report:")
    # print(manager.get_diminishing_returns_report().to_string(index=False))

    print("Cell skipped - uncomment an option above to run additional batches")



STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 45 additional trials
  tvae: 130 additional trials
  ctabganplus: 11 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 45 existing trials


Gen. (-0.08) | Discrim. (-0.33): 100%|██████████| 850/850 [00:41<00:00, 20.25it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5200
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8427
[CHART] Combined Score: 0.6491 (Similarity: 0.5200, Accuracy: 0.8427)
[ctgan] Trial 46: Combined Score: 0.6491 (Similarity: 0.5200, Accuracy: 0.8427) Best Score so far: 0.6986


Gen. (-1.16) | Discrim. (0.07): 100%|██████████| 950/950 [00:47<00:00, 19.84it/s] 


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5351
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8854
[CHART] Combined Score: 0.6752 (Similarity: 0.5351, Accuracy: 0.8854)
[ctgan] Trial 47: Combined Score: 0.6752 (Similarity: 0.5351, Accuracy: 0.8854) Best Score so far: 0.6986


...

Finished training in 409.01688528060913  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6066
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9841
[CHART] Combined Score: 0.7576 (Similarity: 0.6066, Accuracy: 0.9841)
[ctabganplus] Trial 39: Combined Score: 0.7576 (Similarity: 0.6066, Accuracy: 0.9841) Best Score so far: 0.7673


100%|██████████| 800/800 [05:42<00:00,  2.33it/s]


Finished training in 344.25308418273926  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6132
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9731
[CHART] Combined Score: 0.7572 (Similarity: 0.6132, Accuracy: 0.9731)
[ctabganplus] Trial 40: Combined Score: 0.7572 (Similarity: 0.6132, Accuracy: 0.9731) Best Score so far: 0.7673


100%|██████████| 750/750 [05:21<00:00,  2.34it/s]


Finished training in 322.67071557044983  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6267
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9742
[CHART] Combined Score: 0.7657 (Similarity: 0.6267, Accuracy: 0.9742)
[ctabganplus] Trial 41: Combined Score: 0.7657 (Similarity: 0.6267, Accuracy: 0.9742) Best Score so far: 0.7673
  [OK] CTABGAN+: 11 trials in 3827.5s
  Best score: 0.7673

TIME ESTIMATES
  ctgan:
    Avg trial time: 41.8s
    Trials/hour: 86.1
    Time for 50 more trials: 34.8 min
  tvae:
    Avg trial time: 18.8s
    Trials/hour: 191.3
    Time for 50 more trials: 15.7 min
  copulagan:
    Avg trial time: 61.0s
    Trials/hour: 59.0
    Time for 50 more trials: 50.9 min
  ctabgan:
    Avg trial time: 31.3s
    Trials/hour: 115.1
    Time for 50 more trials: 26.1 min
  ctabganplus:
    Avg trial time: 204.4s
    Trials/hour: 17.6
    Time for 50 more trials: 170.3 min
  pategan:
    Avg tr

### 4.6 Save Best Parameters

In [14]:
# Code Chunk ID: CHUNK_4_6_SAVE
# ============================================================================
# SECTION 4.6 - SAVE BEST PARAMETERS TO CSV
# ============================================================================

from src.utils.parameters import save_best_parameters_to_csv

# Extract studies from manager to globals for saving
for model_name, state in manager.model_states.items():
    if state.study is not None:
        globals()[f"{model_name}_study"] = state.study

# Save all best parameters to CSV for Section 5
save_result = save_best_parameters_to_csv(
    scope=globals(),
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER
)

print(f"\nParameters saved: {save_result['success']}")
print(f"Files: {save_result.get('files_saved', [])}")

[SAVE] SAVING BEST PARAMETERS FROM SECTION 4
[FOLDER] Target directory: results/pakistani-diabetes-dataset/2026-03-11/Section-4

[CHART] Processing CTGAN parameters...
[OK] Found CTGAN: 10 parameters, score: 0.7039

[CHART] Processing CTAB-GAN parameters...
[OK] Found CTAB-GAN: 2 parameters, score: 0.7462

[CHART] Processing CTAB-GAN+ parameters...
[OK] Found CTAB-GAN+: 2 parameters, score: 0.7673

[CHART] Processing GANerAid parameters...
[WARNING]  GANerAid: Study variable 'ganeraid_study' not found

[CHART] Processing CopulaGAN parameters...
[OK] Found CopulaGAN: 6 parameters, score: 0.7651

[CHART] Processing TVAE parameters...
[OK] Found TVAE: 8 parameters, score: 0.8020

[CHART] Processing PATE-GAN parameters...
[OK] Found PATE-GAN: 10 parameters, score: 0.4156

[CHART] Processing MEDGAN parameters...
[OK] Found MEDGAN: 11 parameters, score: 0.5269

[OK] Parameters saved: results/pakistani-diabetes-dataset/2026-03-11/Section-4/best_parameters.csv
   - Total parameter entries: 67


## 5 Final Model Comparison with Best Parameters

### 5.1 Train All Models with Best Parameters from Section 4

In [15]:
# Code Chunk ID: CHUNK_5_1_BATCH
# ============================================================================
# SECTION 5.1 - BATCH TRAINING WITH BEST PARAMETERS
# ============================================================================

from src.models.batch_training import train_models_batch_with_best_params

# Train all models with best parameters from Section 4 (checkpoint resumes completed models)
section5_results = train_models_batch_with_best_params(
    data=data,
    target_column=target_column,
    models_to_run=models_to_run,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER,
    scope=globals(),
    verbose=True,
    checkpoint=checkpoint
)

# Show summary of created variables
final_vars = [k for k in globals().keys() if k.endswith('_final')]
print(f"\nFinal synthetic data variables: {sorted(final_vars)}")


SECTION 5.1: BATCH TRAINING WITH BEST PARAMETERS
Models to train: 7
Dataset shape: (912, 19)
Target column: outcome
Samples to generate: 912
Loading parameters from: Section 4
GPU available: NVIDIA A10G (22.1 GB)
Device assignments:
  - CTGAN: cuda
  - TVAE: cuda
  - CopulaGAN: cuda
  - CTABGAN: cuda
  - CTABGAN+: cuda
  - PATE-GAN: cuda
  - MEDGAN: cuda

[1/3] Loading best parameters from Section 4...
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/pakistani-diabetes-dataset/2026-03-11/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 10 parameters
[OK] Loaded CTAB-GAN: 2 parameters
[OK] Loaded CTAB-GAN+: 2 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 8 parameters
[OK] Loaded PATE-GAN: 10 parameters
[OK] Loaded MEDGAN: 11 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 7
   - ctgan: 10 parameters
   - ctabgan: 2 parameters
   - ctabganplus: 2 parameters
   - c

Gen. (-1.18) | Discrim. (-0.01): 100%|██████████| 1000/1000 [00:28<00:00, 34.90it/s]


  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5764
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9013
[CHART] Combined Score: 0.7064 (Similarity: 0.5764, Accuracy: 0.9013)
  [OK] CTGAN completed in 33.75s
  Synthetic data shape: (912, 19)
  Objective score: 0.7064

[2/7] Training TVAE...
--------------------------------------------------
  Device: cuda
  Parameters loaded: 8
    - epochs: 650
    - batch_size: 512
    - learning_rate: 0.0051
    - embedding_dim: 256
    - l2scale: 0.0000
    ... and 4 more
  Training TVAE...
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6798
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9666
[CHART] Combined Score: 0.7945 (Similarity: 0.6798, Accuracy: 0.9666)
  [OK] TVAE completed in 18.81s
  Synthetic data shape: (912,

100%|██████████| 550/550 [00:33<00:00, 16.24it/s]


Finished training in 35.377718687057495  seconds.
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5253
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9693
[CHART] Combined Score: 0.7029 (Similarity: 0.5253, Accuracy: 0.9693)
  [OK] CTABGAN completed in 35.44s
  Synthetic data shape: (912, 19)
  Objective score: 0.7029

[5/7] Training CTABGAN+...
--------------------------------------------------
  Device: cuda
  Parameters loaded: 2
    - epochs: 900
    - batch_size: 64
    - categorical_columns: ['gender', 'rgn', 'his', 'vision', 'dipsia', 'uria', 'neph']
    - target_col: outcome
  Training CTABGAN+...


100%|██████████| 900/900 [06:25<00:00,  2.33it/s]


Finished training in 387.1220591068268  seconds.
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6012
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9759
[CHART] Combined Score: 0.7511 (Similarity: 0.6012, Accuracy: 0.9759)
  [OK] CTABGAN+ completed in 387.21s
  Synthetic data shape: (912, 19)
  Objective score: 0.7511

[6/7] Training PATE-GAN...
--------------------------------------------------
  Device: cuda
  Parameters loaded: 10
    - epochs: 450
    - batch_size: 64
    - num_teachers: 30
    - generator_lr: 0.0002
    - discriminator_lr: 0.0000
    ... and 6 more
  Training PATE-GAN...
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.3169
[ERROR] TRTS (Real->Synthetic) failed: Classification metrics can't handle a mix of continuous and binary targets
[

### 5.2 Batch Evaluation of Optimized Models

In [16]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# ============================================================================

print("SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)

from setup import evaluate_section5_optimized_models

if checkpoint.exists('section_5.2'):
    section5_batch_results = checkpoint.load('section_5.2')['results']
    print("[RESUME] Loaded Section 5.2 evaluation from checkpoint")
    print(f"Models processed: {section5_batch_results['models_processed']}")
    print(f"Results exported to: {section5_batch_results['results_dir']}")
else:
    try:
        section5_batch_results = evaluate_section5_optimized_models(
            section_number=5,
            scope=globals(),
            target_column=TARGET_COLUMN,
            protected_col=NOTEBOOK_CONFIG.get("protected_col"),
            compute_mia=True
        )
        checkpoint.save('section_5.2', {'results': section5_batch_results})
        
        print("\n" + "="*80)
        print("SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
        print("="*80)
        print(f"Models processed: {section5_batch_results['models_processed']}")
        print(f"Results exported to: {section5_batch_results['results_dir']}")
        
    except Exception as e:
        print(f"Section 5.2 batch evaluation failed: {e}")
        import traceback
        traceback.print_exc()

SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 5
[INFO] Dataset: pakistani-diabetes-dataset
[INFO] Target column: outcome
[INFO] Protected column: None (fairness metrics skipped)
[INFO] MIA evaluation: OFF
[INFO] Variable pattern: final
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/pakistani-diabetes-dataset/2026-03-11/Section-5/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.832

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.019
   [CHART] Explained Variance (PC1, PC2): 0.30

### 5.3 Final Summary

In [17]:
# Code Chunk ID: CHUNK_5_3_SUMMARY
# ============================================================================
# SECTION 5.3 - FINAL SUMMARY
# ============================================================================

print("=" * 80)
print("SYNTHETIC DATA GENERATION - FINAL SUMMARY")
print("=" * 80)
print(f"\nDataset: {DATASET_IDENTIFIER}")
print(f"Session: {SESSION_TIMESTAMP}")
print(f"\nResults directories:")
print(f"  Section 2 (EDA): {get_results_path(DATASET_IDENTIFIER, 2)}")
print(f"  Section 3 (Demo): {get_results_path(DATASET_IDENTIFIER, 3)}")
print(f"  Section 4 (HPO): {get_results_path(DATASET_IDENTIFIER, 4)}")
print(f"  Section 5 (Final): {get_results_path(DATASET_IDENTIFIER, 5)}")

# Show staged optimization summary
if 'manager' in dir() and manager is not None:
    print(f"\nStaged Optimization Summary:")
    print("-" * 60)
    summary = manager.get_best_params_summary()
    if len(summary) > 0:
        print(summary[['model', 'best_score', 'total_trials', 'status']].to_string(index=False))

# Show final model performance
if 'section5_results' in dir() and section5_results:
    print(f"\nFinal Model Performance (with optimized parameters):")
    print("-" * 60)
    
    scores = []
    for model_name, result in section5_results.items():
        if result['status'] == 'success':
            score = result.get('objective_score', 0)
            time_taken = result.get('training_time', 0)
            scores.append((model_name, score, time_taken))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    
    for i, (model, score, time_taken) in enumerate(scores, 1):
        print(f"  {i}. {model.upper()}: score={score:.4f}, time={time_taken:.1f}s")
    
    if scores:
        best_model = scores[0][0]
        print(f"\nBest performing model: {best_model.upper()}")
        print(f"Best synthetic data variable: synthetic_{best_model}_final")

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETE")
print("=" * 80)

SYNTHETIC DATA GENERATION - FINAL SUMMARY

Dataset: pakistani-diabetes-dataset
Session: 2026-03-11

Results directories:
  Section 2 (EDA): results/pakistani-diabetes-dataset/2026-03-11/Section-2
  Section 3 (Demo): results/pakistani-diabetes-dataset/2026-03-11/Section-3
  Section 4 (HPO): results/pakistani-diabetes-dataset/2026-03-11/Section-4
  Section 5 (Final): results/pakistani-diabetes-dataset/2026-03-11/Section-5

Staged Optimization Summary:
------------------------------------------------------------
      model  best_score  total_trials    status
      ctgan    0.703889            90 completed
       tvae    0.802032           145 completed
  copulagan    0.765058            15 completed
    ctabgan    0.746152            15 completed
ctabganplus    0.767267            41 completed
    pategan    0.415606            15 completed
     medgan    0.526862            65 completed

Final Model Performance (with optimized parameters):
-----------------------------------------------